In [6]:
import os
from typing import Dict
import numpy as np
import pinocchio as pin
import viser
from viser.extras import ViserUrdf
from yourdfpy import URDF
import time
import simple 


#Βοηθητικες αλλα απαιτητες βιβλιοθηκες απο εξωτερικ αρχεια για αυτο τη simple
from simulation_utils import (
    setPhysicsProperties, #για ιδιοτητες υλικων τριβη ελαστικοτητας
    removeBVHModelsIfAny, #για μετατροπη σε κυρτα σχηματα για πιο γρηγρους υπολογισμους 
    addFloor,#προσθετεις solid εδαφος
    Policy, # ποια policy ακολουθει το ρομποτ επιστρεφε ροπες 
)
from pin_utils import addSystemCollisionPairs
from simulation_args import SimulationArgs

from sim_utils import setupSimulatorFromArgs
#============================================================================

#για τη  viser να βλεπει τις σωστες γωνιες της pinocchio
def map_pin_vis(q):
    joints = np.array((q[1], q[0], q[2], q[3], q[9], q[8], q[10], q[11], q[13], q[12],q[14], q[15], q[4], q[5], q[6], q[7]))
    return joints

#quaternions match
def wxyz_to_xyzw(array): #accepts np.array
    transformed_array = np.concatenate((array[1:],array[0].reshape(1,)))
    return transformed_array

def xyzw_to_wxyz(array):
    transformed_array = np.concatenate((array[-1].reshape(1,),array[:3]))
    return transformed_array


#=====================================================================================================
#Configuration

def build_leap_hand_model(base_dir: str) -> tuple:
    
    urdf_path = os.path.join(base_dir, "leap_description", "robots", "leap_right.urdf")
    mesh_dirs = [base_dir]
    if not os.path.isfile(urdf_path):
        raise FileNotFoundError(
            f"URDF file not found at '{urdf_path}'.  Please update the base_dir."
        )
    return pin.buildModelsFromUrdf(urdf_path, mesh_dirs)


base_dir = "/home/manolis/thesis_workspace/Diplomatiki"
urdf_path = os.path.join(base_dir, "leap_description", "robots", "leap_right.urdf")
mesh_dir = base_dir
model, geom_model, visual_model = build_leap_hand_model(base_dir)

def filename_handler(fname: str) -> str:
    if fname.startswith("package://"):
        rel = fname[len("package://"):]
        return os.path.join(base_dir, rel)
    if not os.path.isabs(fname):
        return os.path.join(os.path.dirname(urdf_path), fname)
    return fname

urdf_obj = URDF.load(
urdf_path,
build_scene_graph=True,
build_collision_scene_graph=True,
load_meshes=True,
load_collision_meshes=True,
mesh_dir=mesh_dir,
filename_handler=filename_handler,
)


urdf_path2 = "/home/manolis/thesis_workspace/Diplomatiki/cube_description/cube.urdf"
model2,geom_model2,visual_model2 = pin.buildModelsFromUrdf(urdf_path2,root_joint = pin.JointModelFreeFlyer(),root_joint_name = "cube_free_flyer")

urdf_obj_2 = URDF.load(
    urdf_path2,
    build_scene_graph=True,
    build_collision_scene_graph=True,
    load_meshes=True,
    load_collision_meshes=True
    )



server = viser.ViserServer()

cube_frame = server.scene.add_frame(
        name='/cube_frame',
        wxyz=(1.0,0.0,0.0,0.0),
        position=(-0.025,0.04,0.13),
        show_axes=False,
        axes_length=0.05,
        axes_radius=0.01
    )

grid = server.scene.add_grid(
        name='/grid',
        position=(0.0,0.0,0.0),
        width=2.0,
        height=2.0
    )

viser_urdf_2 = ViserUrdf(
        server,
        urdf_obj_2,
        root_node_name="/cube_frame",
        load_meshes=True,
        load_collision_meshes=True
    )

viser_urdf_2.show_visual = True
viser_urdf_2.show_collision = False  # μπορείς να το κάνεις True για να βλέπεις collision meshes

robot_frame = server.scene.add_frame(
        name='/robot_frame',
        wxyz=(1.0,0.0,0.0,0.0),
        position=(0.0,0.0,0.0),
        show_axes=False,
        axes_length=0.05,
        axes_radius=0.01
)


viser_urdf = ViserUrdf(
        server,
        urdf_obj,
        root_node_name="/robot_frame",
        load_meshes=True,
        load_collision_meshes=True
)

viser_urdf.show_visual = True
viser_urdf.show_collision = False  # μπορείς να το κάνεις True για να βλέπεις collision meshes

#τελικο μοντελο με κυβο και χερι
f_model, f_geom_model = pin.appendModel(
    model, model2, 
    geom_model, geom_model2, 
    0, pin.SE3.Identity()
)

print("TOTAL JOINTS",f_model.nq)

_,f_visual_model = pin.appendModel(
    model,model2,
    visual_model,visual_model2,
    0,pin.SE3.Identity()
)

#Αρχικοποιηση
# sizes
nq_hand = model.nq
nv_hand = model.nv

q = pin.neutral(f_model)

q[-7:-4] = np.array(cube_frame.position)
q[-4:] = wxyz_to_xyzw(np.array(cube_frame.wxyz))


q_hand0 = q[0:nq_hand]
viser_urdf.update_cfg(map_pin_vis(q_hand0))


#Βοηθητικη συναρτηση για ενημερωση viser
def update_viser_from_q(q_full):
    

    # --- hand joints -> ViserUrdf
    q_hand = q_full[0:nq_hand]  # hand part only
    viser_urdf.update_cfg(map_pin_vis(q_hand))  # only joints

    # --- cube base -> /cube_frame (cube appended after hand)
    q_cube = q_full[nq_hand:nq_hand+7]
    cube_frame.position = tuple(q_cube[0:3])
    cube_frame.wxyz = tuple(xyzw_to_wxyz(q_cube[3:7]))


removeBVHModelsIfAny(f_geom_model) #για απλοποιηση υπολογισμο συγκρουσεων

#addFloor(f_geom_model,f_visual_model) #ισως χρειαστω να το  βγαλω 

q0=q.copy()
v0 = np.zeros(f_model.nv)

addSystemCollisionPairs(f_model, f_geom_model, q0) #ζευγαρια συγκρουσεων

damping_material = "plastic" #επιλογες wood,metal και αλλα
damping_compliance = 0.00

setPhysicsProperties(f_geom_model, material=damping_material, compliance=damping_compliance)

q_des = q0.copy()

for name in model.names:
        if name.startswith("joint_"):
            jid = model.getJointId(name)
            # Get the index in q corresponding to this joint
            idx = model.idx_qs[jid]
            # Many fingers have one dof; assign 80% of the upper limit.
            # If the joint has multiple dofs (unlikely here) this will
            # assign the same target to all of them.
            dof = model.nqs[jid]
            lo = model.lowerPositionLimit[idx : idx + dof]
            hi = model.upperPositionLimit[idx : idx + dof]
            # Choose target at 80% of the range from zero.  If the range
            # is symmetric about zero, this will be 80% of the maximum
            # absolute value.  Adjust factor as needed.
            q_des[idx : idx + dof] = 0.8 * hi




# Configuration των παραμετρων της προσομοιωσης 
horizon = 3000
dt = 0.005

args = SimulationArgs()
args.num_repetitions = 1
args.horizon = horizon
args.dt = dt
args.Kp = 500
args.Kd = 500
args.tol = 0.0
args.tol_rel = 0.0
args.maxit = 200
args.warm_start = 1
args.mu_prox = 1e-4
args.contact_solver = "ADMM"
args.admm_update_rule = "spectral"
#args.max_patch_size = 2
args.max_contacts_per_pair = 4
args.patch_tolerance = 3e-3 #οσο μεγαλυτερο τοσο πιο μακρια σε αποσταση θα ανιχνευεται η συγκροσυση αρα πιο νωρις 
args.safe_distance = 5e-3 #οσο πιο επιθετικς εινια οι κινησεις τοσο πιο γρηγορα τα βλεπει 
# --------------------------------------------------------------------


finger_names = ["fingertip","fingertip_2","fingertip_3","thumb_fingertip"]
ids = []
for name in finger_names:
    ids.append(f_model.getFrameId(name))

print("Starting fist simulation …")
data = f_model.createData()
geom_data = f_geom_model.createData()
simulator = simple.Simulator(f_model, data, f_geom_model, geom_data)
setupSimulatorFromArgs(simulator,geom_data,args)
simulator.reset()

q = q0.copy()
v = v0.copy()
fext = [pin.Force(np.zeros(6)) for _ in range(f_model.njoints)]
#f_model.gravity = pin.Motion(np.zeros(6))

#===========================================================================
finger_names = ["fingertip","fingertip_2","fingertip_3","thumb_fingertip"]
fingertip_frame_ids = []
for name in finger_names:
    fingertip_frame_ids.append(f_model.getFrameId(name))


def get_state(q,v):
    return np.concatenate((q,v)).reshape(-1,1)


#model και data του worker συναρτηση κοστους σε καθε βημα
def q_cost(x, model, data): 
    q = x[:model.nq].reshape(-1)

    pin.forwardKinematics(model, data, q)
    pin.updateFramePlacements(model, data)

    p_cube = q[16:16+3].reshape(3)

    # Οι 4 στόχοι για τα δαχτυλα
    target_index  = p_cube + np.array([ 0.0,  -0.035,  0.01]) 
    target_middle = p_cube + np.array([ 0.035, 0.0,   0.01]) 
    target_ring   = p_cube + np.array([0.0, 0.035,   0.01]) 
    target_thumb  = p_cube + np.array([ -0.035, 0.0,  0.01]) 
    
    targets = [target_index, target_middle, target_ring, target_thumb]

    cost = 0.0
    for i, fid in enumerate(fingertip_frame_ids):
        # Βρίσκει τη θέση του κάθε δαχτύλου
        p_tip = data.oMf[fid].translation
        
        # Υπολογίζει απόσταση από τον ΔΙΚΟ ΤΟΥ στόχο
        dist_sq = np.sum((p_tip - targets[i]) ** 2)
        cost += 1000.0 * dist_sq
        break

    return cost
    
#αυτη ειναι μονο για την τελικη θεση της τροχιας αλλα δεν την χρησιμοποιω
def φ(x):
    q = x[:f_model.nq].reshape(-1)
    # v = x[f_model.nq:].reshape(-1)   # δεν χρειάζεται τώρα

    pin.forwardKinematics(f_model, data, q)
    pin.updateFramePlacements(f_model, data)

    p_cube = q[16:16+3].reshape(3)

    cost = 0.0
    for fid in fingertip_frame_ids:
        p_tip = data.oMf[fid].translation
        cost += np.sum((p_tip - p_cube) ** 2)

    return cost

#και αυτο ειναι απο το πρωτο paper εινια το S running accumulated cost
def running_cost_tilde(x, u, du, R, v_param):
    # u, du are (16,1)
    qx = q_cost(x)

    duT_R_du = float((du.T @ R @ du).squeeze())
    uT_R_du  = float((u.T @ R @ du).squeeze())
    uT_R_u   = float((u.T @ R @ u).squeeze())

    return (
        qx
        + 0.5 * (1.0 - 1.0 / v_param) * duT_R_du
        + uT_R_du
        + 0.5 * uT_R_u
    )

#πολιτικη που επιστρεφει ροπες και κανει neutralize την βαρυτητα
class JointPositionPolicy(Policy):
    def __init__(
        self,
        model: pin.Model,
        q_des: np.ndarray,
        kp: float = 50.0,
        kd: float = 2.0,
        n_actuated: int = 16,
    ):
        self.model = model
        self.kp = kp
        self.kd = kd
        self.n_actuated = n_actuated
        self.nv = model.nv
        self.nq = model.nq
        self.model = model
        self.data = self.model.createData()

        q_des = np.asarray(q_des).reshape(-1)

        # Δέχεται είτε full q_des είτε μόνο τις actuated γωνίες
        if len(q_des) == self.nq:
            self.q_des_full = q_des.copy()
            self.q_des_act = q_des[:self.n_actuated].copy()
        elif len(q_des) == self.n_actuated:
            self.q_des_full = None
            self.q_des_act = q_des.copy()
        else:
            raise ValueError(
                f"q_des must have length {self.nq} (full state) "
                f"or {self.n_actuated} (actuated joints), got {len(q_des)}."
            )

        # Όρια μόνο για το χέρι
        self.effort_limit = model.effortLimit[:self.n_actuated].copy()
        self.velocity_limit = model.velocityLimit[:self.n_actuated].copy()

    def act(
        self,
        simulator: simple.Simulator,
        q: np.ndarray,
        v: np.ndarray,
        dt: float,
    ) -> np.ndarray:
        q = np.asarray(q).reshape(-1)
        v = np.asarray(v).reshape(-1)

        # Τα πρώτα 16 είναι οι actuated αρθρώσεις του χεριού
        q_act = q[:self.n_actuated]
        v_act = v[:self.n_actuated]

        # PD / PI style position tracking
        position_error = self.q_des_act - q_act
        

        v_clip = 3.0   
        v_used = np.clip(v_act, -v_clip, v_clip)

        velocity_error = -v_used
        tau_pd_act = self.kp * position_error + self.kd * velocity_error

        
        tau_grav_full = pin.computeGeneralizedGravity(self.model, self.data, q)
        tau_grav_act = tau_grav_full[:self.n_actuated]

        tau_final_act = tau_grav_act + tau_pd_act

        # effort clipping
        tau_final_act = np.clip(tau_final_act,-self.effort_limit,self.effort_limit)

        
        too_fast_pos = (v_act > self.velocity_limit) & (tau_final_act > 0.0)
        too_fast_neg = (v_act < -self.velocity_limit) & (tau_final_act < 0.0)
        tau_final_act[too_fast_pos | too_fast_neg] = 0.0

        # full torque vector: 16 hand + 6 cube(=0)
        tau_full = np.zeros(self.nv)
        tau_full[:self.n_actuated] = tau_final_act
        return tau_full


(viser) Connection closed (0, 0 total)

╭────── viser (listening *:8085) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8085   │
│   Websocket │ ws://localhost:8085     │
│             ╵                         │
╰───────────────────────────────────────╯

TOTAL JOINTS 23
Num col pairs =  129
Starting fist simulation …


(viser) Connection opened (0, 1 total), 222 persistent messages

In [7]:
import multiprocessing as mp
import numpy as np
from scipy.interpolate import CubicSpline

local_f_model = None
local_data = None
local_f_geom_model = None
local_sim = None
local_policy = None
local_fext = None
local_dt = None
local_args = None
dt = 0.005 # safe για προσομοιωση

def init_worker(f_model, f_geom_model, args, dt):
    global local_f_model, local_f_geom_model, local_sim
    global local_policy, local_fext, local_dt, local_args
    global local_data

    local_f_model = f_model.copy()
    local_f_geom_model = f_geom_model.copy()
    local_dt = dt
    local_args = args

    local_data = local_f_model.createData()
    local_geom_data = local_f_geom_model.createData()

    local_sim = simple.Simulator(
        local_f_model,
        local_data,
        local_f_geom_model,
        local_geom_data
    )
    setupSimulatorFromArgs(local_sim, local_geom_data, local_args)
    local_sim.reset()

    local_policy = JointPositionPolicy(local_f_model,np.zeros(16),kp=0.2,kd=0.005)

    local_fext = [pin.Force(np.zeros(6)) for _ in range(local_f_model.njoints)]

def build_u_from_knots(U_knots, T, u_min, u_max):
    U_knots = np.asarray(U_knots)
    M, nu = U_knots.shape

    knot_times = np.linspace(0, T - 1, M)
    full_times = np.arange(T)

    u_seq = np.zeros((T, nu))

    for j in range(nu):
        cs = CubicSpline(knot_times, U_knots[:, j], bc_type="clamped")
        u_seq[:, j] = cs(full_times)

    u_seq = np.clip(u_seq, u_min, u_max)
    return u_seq

def rollout_from_knots(task):
    global local_f_model, local_sim, local_policy, local_fext, local_dt, local_data

    current_state, U_sample, T, u_min, u_max = task

    x0 = np.asarray(current_state).reshape(-1)
    q = x0[:local_f_model.nq].copy()
    v = x0[local_f_model.nq:].copy()

    u_seq = build_u_from_knots(U_sample, T, u_min, u_max)

    local_sim.reset()

    total_cost = 0.0
    

    for t in range(T):
        q_target = u_seq[t]

        local_policy.q_des_act = q_target
        tau = local_policy.act(local_sim, q, v, local_dt)

        local_sim.step(q, v, tau, local_fext, local_dt)

        q = local_sim.qnew.copy()
        v = local_sim.vnew.copy()

        if not np.all(np.isfinite(q)) or not np.all(np.isfinite(v)):
            return 1e12


        x = np.concatenate((q, v))
        total_cost += q_cost(x,local_f_model,local_data)

    return total_cost

if __name__ == "__main__":
    # Παράμετροι
    K = 30         # samples
    T = 200       # horizon steps
    M = 20       # knots
    nu = 16
    sigma = 0.05
    
    # Πόσα βήματα αντιστοιχούν σε κάθε knot
    steps_per_knot = T // M 
    

    current_state = np.concatenate((q, v))

    u_min = model.lowerPositionLimit[:16].copy()
    u_max = model.upperPositionLimit[:16].copy()

    q_current = q[:16].copy()
    '''
    q_goal = q_des[:16].copy()

    alphas = np.linspace(0.0, 1.0, M).reshape(-1, 1)
    U_nom = (1 - alphas) * q_current + alphas * q_goal #ειναι Μ,16
    '''
    U_nom = np.tile(q_current,(M,1))
    

    num_cores = max(1, mp.cpu_count() - 1)

    #  στο main thread 
    
    main_sim = simple.Simulator(f_model, data, f_geom_model, geom_data)
    setupSimulatorFromArgs(main_sim, geom_data, args)
    main_sim.reset()
    
    main_policy = JointPositionPolicy(f_model, np.zeros(16), kp=0.2, kd=0.005)

    with mp.Pool(processes=num_cores, initializer=init_worker, initargs=(f_model, f_geom_model, args, dt)) as pool:
        
        # MPPI CONTROL LOOP
        for mppi_step in range(100): # Τρέξε τον αλγόριθμο για 100 συνολικά shifts
            print(f"\n MPPI Step {mppi_step} ")
            
            # WARM-UP 
            
            num_updates = 3 if mppi_step == 0 else 1 
            
            for it in range(num_updates):
                noise = sigma * np.random.randn(K, M, nu)
                noise[:, 0, :] = 0.0 # Το πρώτο knot είναι η τρέχουσα θέση

                U_samples = U_nom[None, :, :] + noise
                U_samples = np.clip(U_samples, u_min[None, None, :], u_max[None, None, :])

                tasks = [(current_state, U_samples[k], T, u_min, u_max) for k in range(K)]

                results = pool.map(rollout_from_knots, tasks)
                costs = np.array([r for r in results], dtype=float)

                beta = np.min(costs)
                lam = 0.8 #hyperparameter
                exp_costs = np.exp(-(costs - beta) / lam)
                weights = exp_costs / (np.sum(exp_costs) + 1e-12)

                delta_U = np.sum(weights[:, None, None] * noise, axis=0)
                U_nom = U_nom + delta_U
                
                U_nom[0] = current_state[:16] # Το πρώτο knot καρφωμένο στο τρέχον state
                U_nom = np.clip(U_nom, u_min, u_max)

           
            best_u_seq = build_u_from_knots(U_nom, T, u_min, u_max)
            
            
            q_real = current_state[:f_model.nq].copy()
            v_real = current_state[f_model.nq:].copy()
            
            for t_exec in range(steps_per_knot):
                q_target = best_u_seq[t_exec]
                main_policy.q_des_act = q_target
                tau = main_policy.act(main_sim, q_real, v_real, dt)
                
                
                main_sim.step(q_real, v_real, tau, dt) 
                
                q_real = main_sim.qnew.copy()
                v_real = main_sim.vnew.copy()
                
                update_rate = 4
                if t_exec % update_rate == 0:
                    update_viser_from_q(q_real)
                    time.sleep(dt * update_rate)
            
            
            current_state = np.concatenate((q_real, v_real))
            
            
            U_nom[:-1] = U_nom[1:]
            
            
            U_nom[-1] = U_nom[-2] 
            
            
            U_nom[0] = current_state[:16]
            
            print(f"Εκτελέστηκαν {steps_per_knot} βήματα. Ελάχιστο κόστος πλάνου: {np.min(costs):.4f}")


 MPPI Step 0 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2775.0957

 MPPI Step 1 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2751.8333

 MPPI Step 2 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2698.7140

 MPPI Step 3 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2698.9204

 MPPI Step 4 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2678.5512

 MPPI Step 5 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2721.6410

 MPPI Step 6 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2770.0005

 MPPI Step 7 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2764.9772

 MPPI Step 8 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2629.1265

 MPPI Step 9 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2823.3062

 MPPI Step 10 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2722.5689

 MPPI Step 11 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2812.8973

 MPPI Step 12 
Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 2734.3275

 MPPI Step 13 
Εκτελέστηκαν 10 βήματα. Ελάχιστο

Process ForkPoolWorker-8:


Εκτελέστηκαν 10 βήματα. Ελάχιστο κόστος πλάνου: 6597659.1040

 MPPI Step 71 


KeyboardInterrupt: 